This file focuses on supercon2. Purpose is to organize and clean the database

In [17]:
import numpy as np
import pandas as pd  #to read and work with .csv files
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

In [29]:
df = pd.read_csv('supercon2_v22.12.03.csv')                           #to read the .csv file and store it in a dataframe called df
data = pd.read_csv('supercon2_v22.12.03.csv', header=0)

data.head(10)

,id,rawMaterial,materialId,name,formula,doping,shape,materialClass,fabrication,substrate,...,appliedPressure,section,subsection,hash,title,doi,authors,publisher,journal,year
0,61ba959a7c13e4f711c50c3e,InO x films,1.942078e+09,NaN,InO x,NaN,films,Oxides,NaN,NaN,...,NaN,body,figure,b7d13f231a,Infinite-randomness fixed point of the quantum...,10.1103/physrevb.99.054515,"Nicholas A Lewellyn, Ilana M Percher, J J Nels...",American Physical Society (APS),Physical Review B,2019.0
1,61ba959a0cb3d3a5e6c50c3e,"x = 0.04, 0.05, and 0.06)",1.064204e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,body,paragraph,67f5567e75,Tuning the interplay between nematicity and sp...,10.1038/s41467-018-04471-7,"S-H Baek, Dilip Bhoi, Woohyun Nam, Bumsung Lee...",Springer Science and Business Media LLC,Nature Communications,2018.0
2,61ba959a720b8b3d36c50c3e,amorphous In-O films,-1.411819e+09,NaN,In-O,NaN,films,Alloys,NaN,NaN,...,NaN,body,NaN,e74ca47303,Scaling analysis of the magnetic field-tuned q...,10.1134/1.568304,"V F Gantmakher, M V Golubkov, V T Dolgopolov, ...",Pleiades Publishing Ltd,Journal of Experimental and Theoretical Physic...,1998.0
3,61ba959a6a35f302ddc50c3e,x= 0.108,4.969883e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,body,figure,c8fb7503bc,Doping evolution of antiferromagnetism and tra...,NaN,"Rui Zhang, Dongliang Gong, Xingye Lu, Shiliang...",NaN,NaN,2015.0
4,61ba959a85b12ae79ac50c3e,PrRu 2 Si 2,-5.030707e+08,NaN,PrRu 2 Si 2,NaN,NaN,Alloys,NaN,NaN,...,NaN,header,abstract,e8285dd0b7,Crystal-field interactions in PrRu2Si2,10.1088/0953-8984/12/34/307,"R Michalski, Z Ropka, R J Radwanski",IOP Publishing,Journal of Physics: Condensed Matter,2014.0
5,61ba959a05e6aa6710c50c3e,Lu3Os4Ge13,1.093100e+09,NaN,Lu3Os4Ge13,NaN,NaN,Alloys,NaN,NaN,...,NaN,header,abstract,96a5aef3c7,Superconductivity in Y 3 Ru 4 Ge 13 and Lu 3 O...,NaN,"Om Prakash, A Thamizhavel, S Ramakrishnan",NaN,NaN,2014.0
6,61ba959ba01e2beb09c50c3e,LaMnO 3.04,-7.810355e+08,NaN,LaMnO 3.04,NaN,NaN,Oxides,NaN,NaN,...,NaN,body,paragraph,21ea47f3a7,Ultraslow Polaron Dynamics in Low-Doped Mangan...,10.1103/physrevlett.87.127206,"G Allodi, M Cestelli Guidi, R De Renzi, A Cane...",American Physical Society (APS),Physical Review Letters,2014.0
7,61ba959ba01e2beb09c50c3f,doping con- centrations 0.06 < x ≤ 0.1,-6.308684e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,body,paragraph,21ea47f3a7,Ultraslow Polaron Dynamics in Low-Doped Mangan...,10.1103/physrevlett.87.127206,"G Allodi, M Cestelli Guidi, R De Renzi, A Cane...",American Physical Society (APS),Physical Review Letters,2014.0
8,61ba959ba01e2beb09c50c40,La 0.8 Ca 0.2 MnO 3.015,-2.625461e+08,NaN,La 0.8 Ca 0.2 MnO 3.015,NaN,NaN,Oxides,NaN,NaN,...,NaN,body,paragraph,21ea47f3a7,Ultraslow Polaron Dynamics in Low-Doped Mangan...,10.1103/physrevlett.87.127206,"G Allodi, M Cestelli Guidi, R De Renzi, A Cane...",American Physical Society (APS),Physical Review Letters,2014.0
9,61ba959ba01e2beb09c50c41,LaMnO 3.06,6.122870e+07,NaN,LaMnO 3.06,NaN,NaN,Oxides,NaN,NaN,...,NaN,body,figure,21ea47f3a7,Ultraslow Polaron Dynamics in Low-Doped Mangan...,10.1103/physrevlett.87.127206,"G Allodi, M Cestelli Guidi, R De Renzi, A Cane...",American Physical Society (APS),Physical Review Letters,2014.0


Step 1: get rid of NAN values in appliedPressure row and ambient variations

In [19]:
print(data.columns)                #print the column names of the dataframe to understand the structure of the data

data_clean = data.dropna(subset=['appliedPressure'])             #to remove rows with missing values in the 'appliedPressure' column, remove NaN

# See unique pressure values
#print(data_clean['appliedPressure'].unique())

# Remove ambient variations
data_clean = data_clean[
    ~data_clean['appliedPressure'].astype(str)
    .str.contains('ambient', case=False, na=False)
]


data_clean = data_clean[                                     #Keep only rows that contain a number
    data_clean['appliedPressure'].astype(str)
    .str.contains(r'\d', na=False)
]




valid_units = r'GPa|kbar|MPa|bar|atm|Torr|torr' 

data_clean = data_clean[                                  #keep only rows that contain a unit
    data_clean['appliedPressure'].astype(str)
    .str.contains(valid_units, case=False, na=False)
]

print(data_clean['appliedPressure'].unique()[:])

Index(['id', 'rawMaterial', 'materialId', 'name', 'formula', 'doping', 'shape',
       'materialClass', 'fabrication', 'substrate', 'variables',
       'criticalTemperature', 'criticalTemperatureMeasurementMethod',
       'appliedPressure', 'section', 'subsection', 'hash', 'title', 'doi',
       'authors', 'publisher', 'journal', 'year'],
      dtype='object')
['9 GPa' '2-6 GPa' '7 kbar' '460 GPa' '1 GPa' '5.4 GPa' '2.5 GPa' '20 GPa'
 '150 GPa' '190 GPa' '15 GPa' '25 kbar' '1.48 GPa' '2.6 GPa' '20-30 GPa'
 '0.97GPa pressure' '1GPa' '1.68GPa' '0.35GPa' '0GPa' '1.5GPa' '0.55GPa'
 '1.92 GPa' '6 GPa' '32−35 GPa' '8−10 GPa' '120 GPa' '300 GPa' '18 GPa'
 '5 GPa' '125 GPa' '2.0 GPa' '1.8 and 2.0 GPa' '10 GPa' '8.9 GPa'
 '0.5 GPa' '5kbar' '40-60kbar' '10 GPa pressure' '17 kbar' '11 and 13 GPa'
 '2 GPa' '200 GPa' '155 GPa' '207 GPa' '96 GPa' '250 GPa' '21.5 GPa'
 '3 GPa' '16 GPa' '6 kbar' '17.5, 11, and 14 kbar' '4 GPa' '7.5GPa'
 '40 GPa' '0.7 GPa' '2 × 10 −2 mbar' '2.7 GPa' '210 GPa' '285 GPa'

step 2: convert these pressures to the same units (GPa of Kbar)

In [20]:
import re

#This functiontakes one cell from the appliedPressure column and returns a number in GPa
def convert_to_gpa(text):
    t = str(text).lower()                              #Convert to string and lowercase for easier processing
    numbers = re.findall(r'\d+(?:\.\d+)?', t)           #Extract the first number (integer or decimal) from the text
    if not numbers:
        return np.nan
    value = float(numbers[0])                             #take the first number found and convert it to a float (at least for now), may change it later

    if 'gpa' in t:
        return value
    elif 'kbar' in t:
        return value * 0.1
    elif 'mpa' in t:
        return value * 0.001
    elif 'atm' in t:
        return value * 0.000101325
    elif 'torr' in t:
        return value * 1.33322e-7
    elif 'bar' in t:
        return value * 1e-4
    else:
        return np.nan

data_clean['pressure_GPa'] = data_clean['appliedPressure'].apply(convert_to_gpa)

# Remove anything that failed conversion
data_clean = data_clean.dropna(subset=['pressure_GPa'])                   #created a new column called 'pressure_GPa' by applying the convert_to_gpa function to the 'appliedPressure' column, and then removed any rows where the conversion failed (i.e., where 'pressure_GPa' is NaN)

print(data_clean[['appliedPressure', 'pressure_GPa']])

                                         appliedPressure  pressure_GPa
45                                                 9 GPa  9.000000e+00
118                                              2-6 GPa  2.000000e+00
154                                               7 kbar  7.000000e-01
215                                              460 GPa  4.600000e+02
264                                                1 GPa  1.000000e+00
267                                                1 GPa  1.000000e+00
355                                              5.4 GPa  5.400000e+00
357                                              2.5 GPa  2.500000e+00
358                                               20 GPa  2.000000e+01
359                                               20 GPa  2.000000e+01
378                                              150 GPa  1.500000e+02
379                                              190 GPa  1.900000e+02
482                                               15 GPa  1.500000e+01
503   

Printing results

In [21]:
pd.set_option('display.max_rows', None)
print(data_clean.loc[:, ['appliedPressure', 'pressure_GPa']])

                                         appliedPressure  pressure_GPa
45                                                 9 GPa  9.000000e+00
118                                              2-6 GPa  2.000000e+00
154                                               7 kbar  7.000000e-01
215                                              460 GPa  4.600000e+02
264                                                1 GPa  1.000000e+00
267                                                1 GPa  1.000000e+00
355                                              5.4 GPa  5.400000e+00
357                                              2.5 GPa  2.500000e+00
358                                               20 GPa  2.000000e+01
359                                               20 GPa  2.000000e+01
378                                              150 GPa  1.500000e+02
379                                              190 GPa  1.900000e+02
482                                               15 GPa  1.500000e+01
503   

In [31]:
#data_clean[['id', 'rawMaterial', 'materialId','formula', 'materialClass', 'variables', 'appliedPressure', 'pressure_GPa', 'criticalTemperature']]      #uncomment this line to see all rows of the cleaned dataframe with the specified columns, including the new 'pressure_GPa' column
data_clean[['id', 'rawMaterial', 'materialId','formula', 'materialClass', 'variables', 'appliedPressure', 'pressure_GPa', 'criticalTemperature']].head(50)




,id,rawMaterial,materialId,formula,materialClass,variables,appliedPressure,pressure_GPa,criticalTemperature
45,61ba959de4552e90a8c50c43,FeSe 1−x,-1.050844e+09,FeSe 1−x,"Iron-chalcogenides, Chalcogenides",NaN,9 GPa,9.00,37 K
118,61ba95a185fc223534c50c43,BaFe 2 As 2,-4.731701e+08,BaFe 2 As 2,"Iron-pnictides, Pnictides",NaN,2-6 GPa,2.00,28 K
154,61ba95a45cc438644dc50c43,Cs 3 C 60,3.105467e+08,Cs 3 C 60,Carbides,NaN,7 kbar,0.70,38 K
215,61ba95aa783c1e8128c50c53,metallic hydrogen,4.990380e+07,NaN,NaN,NaN,460 GPa,460.00,234 K
264,61ba95ae783c1e8128c50c56,BaNi 2 P 2 single crystals,-2.531872e+08,BaNi 2 P 2,Pnictides,NaN,1 GPa,1.00,3 K
267,61ba95ae783c1e8128c50c59,BaNi 2 P 2,1.734361e+09,BaNi 2 P 2,Pnictides,NaN,1 GPa,1.00,3 K
355,61ba95b514d91e0862c50c56,CeCoGe 3,-7.490919e+08,CeCoGe 3,Alloys,NaN,5.4 GPa,5.40,0.7 K
357,61ba95b514d91e0862c50c58,CeIrSi 3,-4.382956e+07,CeIrSi 3,Alloys,NaN,2.5 GPa,2.50,1.6 K
358,61ba95b514d91e0862c50c59,CeIrGe 3,8.464833e+08,CeIrGe 3,Alloys,NaN,20 GPa,20.00,1.5 K
359,61ba95b514d91e0862c50c5a,CeIrGe3,9.612450e+07,CeIrGe3,Alloys,NaN,20 GPa,20.00,1.5 K
